## Download Dataset from Roboflow Universe

RF-DETR expects the dataset to be in COCO format. Divide your dataset into three subdirectories: `train`, `valid`, and `test`. Each subdirectory should contain its own `_annotations.coco.json` file that holds the annotations for that particular split, along with the corresponding image files. Below is an example of the directory structure:

```
dataset/
├── train/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
├── valid/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
└── test/
    ├── _annotations.coco.json
    ├── image1.jpg
    ├── image2.jpg
    └── ... (other image files)
```

[Roboflow](https://roboflow.com/annotate) allows you to create object detection datasets from scratch or convert existing datasets from formats like YOLO, and then export them in COCO JSON format for training. You can also explore [Roboflow Universe](https://universe.roboflow.com/) to find pre-labeled datasets for a range of use cases.

In [1]:
import os
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY"))
project = rf.workspace("fyp-vfrgn").project("7-classes-vehicle")
version = project.version(1)
dataset = version.download("coco")

loading Roboflow workspace...
loading Roboflow project...


### EDA

In [2]:
import json
import os
from collections import Counter

# Set your dataset directory path
dataset_dir = "7-classes-vehicle-1"

def analyze_coco_text_only(dataset_path, split_name):
    json_path = os.path.join(dataset_path, split_name, "_annotations.coco.json")
    
    if not os.path.exists(json_path):
        print(f"⚠️ Annotation file not found for {split_name}: {json_path}")
        return None, None
        
    with open(json_path, 'r') as f:
        data = json.load(f)
        
    # Map category IDs to names
    cat_map = {c['id']: c['name'] for c in data['categories']}
    
    # Count classes
    class_counts = Counter([cat_map[ann['category_id']] for ann in data['annotations']])
    
    # Get image dimensions
    img_dims = [(img['width'], img['height']) for img in data['images']]
    
    return class_counts, img_dims

splits = ['train', 'valid', 'test']
all_sizes = []

print("=" * 60)
print(f"{'DATASET ANALYSIS':^60}")
print("=" * 60)

for split in splits:
    counts, sizes = analyze_coco_text_only(dataset_dir, split)
    
    if counts:
        all_sizes.extend(sizes)
        total_anns = sum(counts.values())
        
        print(f"\n📂 SPLIT: {split.upper()}")
        print(f"   Total Images:      {len(sizes)}")
        print(f"   Total Annotations: {total_anns}")
        print("-" * 40)
        print(f"   {'CLASS NAME':<25} | {'COUNT':<10}")
        print("-" * 40)
        
        # Print counts for each class, sorted by count descending
        for class_name, count in counts.most_common():
            print(f"   {class_name:<25} | {count:<10}")
        print("-" * 40)

# --- Image Size Analysis (Text Only) ---
if all_sizes:
    widths, heights = zip(*all_sizes)
    avg_w = sum(widths) / len(widths)
    avg_h = sum(heights) / len(heights)
    
    print("\n" + "=" * 60)
    print(f"{'IMAGE RESOLUTION STATS':^60}")
    print("=" * 60)
    print(f"   Resolution Range: {min(widths)}x{min(heights)} to {max(widths)}x{max(heights)}")
    print(f"   Average Resolution: {avg_w:.0f}x{avg_h:.0f}")
    print("=" * 60)

                      DATASET ANALYSIS                      

📂 SPLIT: TRAIN
   Total Images:      5819
   Total Annotations: 33129
----------------------------------------
   CLASS NAME                | COUNT     
----------------------------------------
   Pickup                    | 11649     
   Sedan                     | 11522     
   Truck                     | 5527      
   Suv                       | 2267      
   Van                       | 1621      
   Motorcycle                | 416       
   Bus                       | 127       
----------------------------------------

📂 SPLIT: VALID
   Total Images:      727
   Total Annotations: 4264
----------------------------------------
   CLASS NAME                | COUNT     
----------------------------------------
   Pickup                    | 1469      
   Sedan                     | 1453      
   Truck                     | 759       
   Suv                       | 317       
   Van                       | 199       
   Mot

## Train RF-DETR on custom dataset

### Choose the right `batch_size`

Different GPUs have different amounts of VRAM (video memory), which limits how much data they can handle at once during training. To make training work well on any machine, you can adjust two settings: `batch_size` and `grad_accum_steps`. These control how many samples are processed at a time. The key is to keep their product equal to 16 — that’s our recommended total batch size. For example, on powerful GPUs like the A100, set `batch_size=16` and `grad_accum_steps=1`. On smaller GPUs like the T4, use `batch_size=4` and `grad_accum_steps=4`. We use a method called gradient accumulation, which lets the model simulate training with a larger batch size by gradually collecting updates before adjusting the weights.

In [3]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())

True
12.6
1


In [4]:
from rfdetr.datasets.aug_config import AUG_AGGRESSIVE  # Built-in option

# custom_aug_config = {
#     "HorizontalFlip": {"p": 0.5},
#     "ShiftScaleRotate": {
#         "shift_limit": 0.05,
#         "scale_limit": 0.2,       # Increased from 0.15 — helps with vehicle size variance
#         "rotate_limit": 10,
#         "p": 0.5
#     },
#     "RandomBrightnessContrast": {
#         "brightness_limit": 0.2,  # Increased from 0.1 — simulates day/night transitions
#         "contrast_limit": 0.2,   # Increased from 0.1
#         "p": 0.4                  # Increased from 0.3
#     },
#     "OneOf": [                    # New: color variation (supported in 1.5.1)
#         {"HueSaturationValue": {"hue_shift_limit": 10, "sat_shift_limit": 20, "val_shift_limit": 20, "p": 0.5}},
#         {"RGBShift": {"r_shift_limit": 15, "g_shift_limit": 15, "b_shift_limit": 15, "p": 0.5}},
#     ],
# }

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [5]:
from rfdetr import RFDETRMedium

model = RFDETRMedium()
history = []

def callback2(data):
	history.append(data)

model.callbacks["on_fit_epoch_end"].append(callback2)

model.train(
    dataset_dir=dataset.location,
    resolution=576,
    epochs=150,
    batch_size=4,
    grad_accum_steps=4,
    lr=5e-5,
    lr_encoder=7.5e-5,
    weight_decay=1e-4,
    aug_config=AUG_AGGRESSIVE,
    cls_loss_coef=2.0,
    lr_scheduler="cosine",
    warmup_epochs=1.0,
    drop_path=0.1,
    save_dataset_grids=True,
    progress_bar=True,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_min_delta=0.001,
    checkpoint_interval=5,
    output_dir='output_medium'
)

[2026-04-12 21:23:51] [INFO] rf-detr - File rf-detr-medium.pth already exists with correct MD5 hash.


[2026-04-12 21:23:51] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-12 21:23:51] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-04-12 21:23:52] [INFO] rf-detr - File rf-detr-medium.pth already exists with correct MD5 hash.


W0412 21:23:57.339000 12252 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
[2026-04-12 21:23:58] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-04-12 21:23:58] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-04-12 21:23:59] [INFO] rf-detr - File rf-detr-medium.pth already exists with correct MD5 hash.


...

Val — Overall Metrics                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.5970 │ 0.8943 │ 0.6811 │ 0.7125 │ 0.8305 │ 0.8046 │ 0.8604 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘

                    Val — Per-class Metrics                     
┏━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Class      ┃ AP 50:95 ┃     AR ┃     F1 ┃ Precision ┃ Recall ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ Bus        │   0.6885 │ 0.7389 │ 0.9143 │    0.9412 │ 0.8889 │
│ Motorcycle │   0.4661 │ 0.5816 │ 0.8155 │    0.7778 │ 0.8571 │
│ Pickup     │   0.6222 │ 0.7183 │ 0.8552 │    0.8158 │ 0.8986 │
│ Sedan      │   0.5999 │ 0.7086 │ 0.8386 │    0.7828 │ 0.9030 │
│ Suv        │   0.4886 │ 0.7268 │ 0.6880 │    0.6981 │ 0.6782 │
│ Truck      │   0.6856 │ 0.7838 │ 0.8533 │    0.8049 │ 0.9078 │
│ Van        │   0.6278 │ 0.7296 │ 0.8489 │    0.8119 │ 0.8894 │
└────────────┴──────────┴────────┴────────┴───────────┴────────┘

Epoch 102: 100%|██████████| 1456/1456 [08:39<00:00,  2.80it/s, train/lr=1.13e-5, train/lr_min=3.65e-7, train/lr_max=1.13e-5, val/loss=5.400, val/mAP_50_95=0.597, val/mAP_50=0.894, val/ema_mAP_50_95=0.599, val/F1=0.831, train/loss=5.540]

Monitored metric __rfdetr_effective_map__ did not improve in the last 20 records. Best score: 0.598. Signaling Trainer to stop.


[2026-04-13 12:32:52] [INFO] rf-detr - Best EMA mAP improved to 0.5987 (epoch 102)
Epoch 102: 100%|██████████| 1456/1456 [08:42<00:00,  2.79it/s, train/lr=1.13e-5, train/lr_min=3.65e-7, train/lr_max=1.13e-5, val/loss=5.400, val/mAP_50_95=0.597, val/mAP_50=0.894, val/ema_mAP_50_95=0.599, val/F1=0.831, train/loss=5.530]
[2026-04-13 12:32:59] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.5983, ema=0.5987)


In [6]:
# from PIL import Image

# Image.open("output/metrics_plot.png")

## Evaluate Fine-tuned RF-DETR Model

Before benchmarking the model, we need to load the best saved checkpoint. To ensure it fits on the GPU, we first need to free up GPU memory. This involves deleting any remaining references to previously used objects, triggering Python’s garbage collector, and clearing the CUDA memory cache.

In [7]:
import gc
import torch
import weakref

def cleanup_gpu_memory(obj=None, verbose: bool = False):

    if not torch.cuda.is_available():
        if verbose:
            print("[INFO] CUDA is not available. No GPU cleanup needed.")
        return

    def get_memory_stats():
        allocated = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        return allocated, reserved

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[Before] Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

    # Ensure we drop all strong references
    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("[WARNING] Object not fully garbage collected yet.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[After]  Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

In [8]:
cleanup_gpu_memory(model, verbose=True)

[Before] Allocated: 128.34 MB | Reserved: 232.00 MB
[WARNING] Object not fully garbage collected yet.
[After]  Allocated: 0.00 MB | Reserved: 0.00 MB


We load the best-performing model from the `checkpoint_best_total.pth` file using the `RFDETRMedium` class. This checkpoint contains the trained weights from our most successful training run. After loading, we call `optimize_for_inference()`, which prepares the model for efficient inference.

In [ ]:
from rfdetr import RFDETRMedium
model = RFDETRMedium(pretrain_weights="output_medium/checkpoint_best_total.pth")
model.optimize_for_inference()

In [11]:
import supervision as sv

ds = sv.DetectionDataset.from_coco(
    images_directory_path=f"{dataset.location}/test",
    annotations_path=f"{dataset.location}/test/_annotations.coco.json",
)

In [12]:
import supervision as sv
import numpy as np
from PIL import Image
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision

targets = []
predictions = []

for path, image, annotations in tqdm(ds):
    image = Image.open(path)
    detections = model.predict(image, threshold=0)

    # Remove image-level metadata that supervision can't per-detection index
    if 'source_shape' in detections.data:
        del detections.data['source_shape']

    targets.append(annotations)
    predictions.append(detections)

100%|██████████| 728/728 [00:24<00:00, 30.22it/s]


In [13]:
map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()
print(map_result)

Average Precision (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.584
Average Precision (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.881
Average Precision (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.695
Average Precision (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.306
Average Precision (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.583
Average Precision (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.715
